In [1]:
%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3

# %matplotlib qt
%matplotlib inline
import matplotlib.pyplot as plt

from pathlib import Path
import spikeinterface.extractors as se
import spikeinterface.full as si
from neuropy.core.session.init_from_raw_data import windows_to_wsl_path_if_needed, find_first_file_rglob

from spikeinterface_pipeline import BapunSessionConfig, CurationConfig, resolve_session_paths, load_bapun_recording, load_phy_sorting, load_spykingcircus_sorting, run_phy_curation_pipeline, compute_qm_labels


Automatic pdb calling has been turned OFF


/home/halechr/FastData/Bapun/RatS/NeuroPy/neuropy/utils/mixins/time_slicing.py:404: UserWarning: registration of accessor <class 'neuropy.utils.mixins.time_slicing.TimePointEventAccessor'> under name 'time_point_event' for type <class 'pandas.core.frame.DataFrame'> is overriding a preexisting attribute with the same name.
  @pd.api.extensions.register_dataframe_accessor("time_point_event")


In [2]:
## Bapun Format:
# basedir = Path('/home/halechr/FastData/Bapun/RatS/Day1Openfield') # Linux
# basedir = Path('/nfs/turbo/umms-kdiba/Data/Bapun/RatS/Day1Openfield') # Greatlakes
# basedir = windows_to_wsl_path_if_needed(Path('W:/Data/Bapun/RatS/Day1Openfield')) # Windows
# basedir = Path("/mnt/w/Data/Bapun/RatS/Day1Openfield") # Apogee WSL
# n_channels: int = 195
# dat_file_sampling_rate: int = 30000
# eeg_sampling_rate: int = 1250
# basename: str = 'RatS-Day1Openfield'
# excluded_data_datetimes = ['2020-11-25_10-24-24', '2020-11-25_15-06-02']


# basedir = Path('/home/halechr/FastData/Bapun/RatS/Day4Openfield') # Linux
basedir = Path('/media/halechr/MoverNew/Data/Bapun/RatS/Day4Openfield') # Linux
# basedir = Path('/nfs/turbo/umms-kdiba/Data/Bapun/RatS/Day4Openfield') # Greatlakes
# basedir = windows_to_wsl_path_if_needed(Path('W:/Data/Bapun/RatS/Day4Openfield')) # Windows
# basedir = Path("/mnt/w/Data/Bapun/RatS/Day4Openfield") # Apogee WSL

n_channels: int = 195
dat_file_sampling_rate: int = 30000
eeg_sampling_rate: int = 1250
basename: str = 'RatS-Day4Openfield'
excluded_data_datetimes = []

In [3]:
session = BapunSessionConfig(basedir=basedir, basename=basename, n_channels=n_channels, dat_file_sampling_rate=dat_file_sampling_rate, eeg_sampling_rate=eeg_sampling_rate, excluded_data_datetimes=excluded_data_datetimes)
paths = resolve_session_paths(session)
xml_path = paths.xml_path
concatenated_dat_file = paths.concatenated_dat_file
print(f'xml_path: {xml_path}')
print(f"concatenated_dat_file: {concatenated_dat_file.as_posix()}")
assert concatenated_dat_file.exists(), f"concatenated_dat_file: {concatenated_dat_file} does not exist."


xml_path: /media/halechr/MoverNew/Data/Bapun/RatS/Day4Openfield/RatS-Day4Openfield.xml
concatenated_dat_file: /media/halechr/MoverNew/Data/Bapun/RatS/Day4Openfield/spyk-circ/RatS-Day4Openfield.dat


In [4]:
spiking_circus_loaded = load_spykingcircus_sorting(paths)
spiking_circus_loaded


SpykingCircusSortingExtractor: 258 units - 1 segments - 30.0kHz

In [5]:
sorting = load_phy_sorting(paths)
sorting


PhySortingExtractor: 257 units - 1 segments - 30.0kHz

In [ ]:
n_channels

In [6]:
recording, recording_filtered = load_bapun_recording(session, paths)
recording


ChannelSliceRecording: 175 channels - 30.0kHz - 1 segments - 230,897,884 samples 
                       7,696.60s (2.14 hours) - int16 dtype - 75.26 GiB

In [7]:
recording_filtered


BandpassFilterRecording: 175 channels - 30.0kHz - 1 segments - 230,897,884 samples 
                         7,696.60s (2.14 hours) - int16 dtype - 75.26 GiB

## Run Manually (2026-06-02)

In [ ]:
sorting = spiking_circus_loaded

job_kwargs = dict(n_jobs=-1, progress_bar=True, chunk_duration="1s")

sorting_analyzer_folder: Path = windows_to_wsl_path_if_needed(basedir.joinpath("spyk-circ", f"{basename}-sorting_analyzer").resolve())
sorting_analyzer = si.create_sorting_analyzer(sorting, recording_filtered, format="binary_folder", folder=sorting_analyzer_folder.as_posix())
sorting_analyzer.compute("random_spikes", method="uniform", max_spikes_per_unit=500)
sorting_analyzer.compute("waveforms", **job_kwargs)
sorting_analyzer.compute("templates", **job_kwargs)
sorting_analyzer.compute("noise_levels")
sorting_analyzer.compute("unit_locations", method="monopolar_triangulation")
sorting_analyzer.compute("isi_histograms")
sorting_analyzer.compute("correlograms", window_ms=100, bin_ms=5.)
sorting_analyzer.compute("principal_components", n_components=3, mode='by_channel_global', whiten=True, **job_kwargs)
sorting_analyzer.compute("quality_metrics", metric_names=["snr", "firing_rate"])
sorting_analyzer.compute("template_similarity")
sorting_analyzer.compute("spike_amplitudes", **job_kwargs)

In [ ]:
from spikeinterface.sorters import installed_sorters, run_sorter

installed_sorters() # ['kilosort4', 'lupin', 'simple', 'spykingcircus2', 'tridesclous2']

sorting_outputs_folder: Path = windows_to_wsl_path_if_needed(basedir.joinpath("SORTING").resolve()) # , f"{basename}-sorting_analyzer"
sorting_outputs_folder.mkdir(exist_ok=True)
print(f'sorting_outputs_folder: {sorting_outputs_folder}')


In [ ]:

# run SpykingCircus2
sorting_SC2 = run_sorter(sorter_name="spykingcircus2", recording=recording, folder=sorting_outputs_folder.joinpath("folder_SC2"), remove_existing_folder=True, delete_output_folder=True)


In [ ]:
# run Tridesclous
TDC_Output_Path: Path = sorting_outputs_folder.joinpath("folder_TDC")
sorting_TDC = run_sorter(sorter_name="tridesclous", recording=recording_filtered, folder=TDC_Output_Path)
sorting_TDC

In [ ]:
TDC_Final_Sorting_Output_Path: Path = TDC_Output_Path.joinpath('tridescloud_sorting_output')
sorting_TDC.save(folder=TDC_Final_Sorting_Output_Path)


In [ ]:
sorting_TDC.sorting_info

In [ ]:
# run Kilosort4
sorting_KS2_5 = run_sorter(sorter_name="kilosort4", recording=recording, folder=sorting_outputs_folder.joinpath("folder_KS4"))

In [ ]:
sorting = run_sorter(sorter_name="spykingcircus2", recording=recording, folder="/folder_SC2tmp/folder")

In [ ]:
# run SpykingCircus2
sorting_SC2 = run_sorter(sorter_name="spykingcircus2", recording=recording, folder=sorting_outputs_folder.joinpath("folder_SC2"))
sorting_SC2

In [ ]:
sorting_SC2

# Quality Metrics Tutorial

After spike sorting, you might want to validate the 'goodness' of the sorted units. This can be done using the
:code:`qualitymetrics` submodule, which computes several quality metrics of the sorted units.


In [ ]:
import spikeinterface.core as si
from spikeinterface.metrics import (
    compute_snrs,
    compute_presence_ratios,
    compute_isi_violations,
)

First, let's generate a simulated recording and sorting



In [ ]:
# sorting = sorting_TDC
# sorting_analyzer = sorting_TDC.to_shared_memory_sorting()

sorting_analyzer = si.create_sorting_analyzer(sorting=sorting, recording=recording_filtered)
sorting_analyzer.compute(['noise_levels','random_spikes','waveforms','templates','spike_locations','spike_amplitudes','correlograms','principal_components','quality_metrics','template_metrics'])


In [ ]:
sorting_analyzer.compute('template_metrics', include_multi_channel_metrics=True) 

# recording, sorting = si.generate_ground_truth_recording()
# print(recording)
# print(sorting)

In [ ]:
all_metric_names = list(sorting_analyzer.get_extension('quality_metrics').get_data().keys()) + list(sorting_analyzer.get_extension('template_metrics').get_data().keys())
print(set(model.feature_names_in_).issubset(set(all_metric_names)))

## Create SortingAnalyzer

For quality metrics we need first to create a :code:`SortingAnalyzer`.



The :code:`spikeinterface.qualitymetrics` submodule has a set of functions that allow users to compute
metrics in a compact and easy way. To compute a single metric, one can simply run one of the
quality metric functions as shown below. Each function has a variety of adjustable parameters that can be tuned.



In [ ]:
## INPUTS: sorting_analyzer
presence_ratios = compute_presence_ratios(sorting_analyzer)
print(presence_ratios)
isi_violation_ratio, isi_violations_count = compute_isi_violations(sorting_analyzer)
print(isi_violation_ratio)
snrs = compute_snrs(sorting_analyzer)
print(snrs)

In [ ]:
import spikeinterface.curation as sc

labels = sc.model_based_label_units(
    sorting_analyzer = sorting_analyzer,
    repo_id = "SpikeInterface/toy_tetrode_model",
    trusted = ['numpy.dtype']
)

print(labels)

To compute more than one metric at once, we can use the :code:`SortingAnalyzer.compute("quality_metrics")`
function and indicate which metrics we want to compute. Then we can retrieve the results using the :code:`get_data()`
method as a ``pandas.DataFrame``.



In [ ]:
metrics_ext = sorting_analyzer.compute(
    "quality_metrics",
    metric_names=["presence_ratio", "snr", "amplitude_cutoff"],
    metric_params={
        "presence_ratio": {"bin_duration_s": 2.0},
    }
)
metrics = metrics_ext.get_data()
print(metrics)

Some metrics are based on the principal component scores, so the extension
must be computed before. For instance:



In [ ]:
sorting_analyzer.compute("principal_components", n_components=5, mode="by_channel_global", whiten=True)

metrics_ext = sorting_analyzer.compute(
    "quality_metrics",
    metric_names=[
        "mahalanobis",
        "d_prime",
    ],
)
metrics = metrics_ext.get_data()
print(metrics)

# Trained Models

In [ ]:
from spikeinterface.curation import model_based_label_units

labels_and_probabilities = model_based_label_units(
    sorting_analyzer = sorting_analyzer,
    repo_id = "SpikeInterface/toy_tetrode_model",
    trust_model = True
)

# Apply to Phy Data

In [10]:
# Curation parameters (tune here)
PROB_THRESHOLD_HIGH: float = 0.8
PROB_THRESHOLD_DEFAULT: float = 0.65
PROB_CUTOFFS: list[float] = [0.5, 0.65, 0.8]
GOOD_UNIT_STRATEGY: str = "sua_relaxed_prob"
allow_overwrite: bool = True


In [11]:
curation = CurationConfig(strategy=GOOD_UNIT_STRATEGY, prob_default=PROB_THRESHOLD_DEFAULT, prob_high=PROB_THRESHOLD_HIGH, analyzer_overwrite="if_missing")
result = run_phy_curation_pipeline(session, curation, patch_pandas_compat=False)
sorting = result.sorting
sorting_analyzer = result.sorting_analyzer
analyzer_neural = result.analyzer_neural
all_labels = result.all_labels
comparison = result.comparison
good_units = result.good_units
sorting_curated = result.sorting_curated
print(f"Strategy: {GOOD_UNIT_STRATEGY} -> {len(good_units)} good units")
print(all_labels["prediction"].value_counts())
print(comparison.groupby(["phy_label", "prediction"]).size())


estimate_sparsity (no parallelization):   0%|          | 0/7697 [00:00<?, ?it/s]

/home/halechr/FastData/Bapun/RatS/spikeinterface/src/spikeinterface/core/sortinganalyzer.py:371: UserWarning: Your sorting has spikes with samples times greater than your recording length. These spikes have been removed.
  warnings.warn(


noise_level (workers: 8 processes fork):   0%|          | 0/20 [00:00<?, ?it/s]

compute_waveforms (workers: 8 processes fork):   0%|          | 0/1925 [00:00<?, ?it/s]

/home/halechr/FastData/Bapun/RatS/spikeinterface/src/spikeinterface/postprocessing/correlograms.py:737: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  numba.set_num_threads(num_threads)


...

Projecting waveforms:   0%|          | 0/257 [00:00<?, ?it/s]

/home/halechr/FastData/Bapun/RatS/spikeinterface/src/spikeinterface/metrics/template/template_metrics.py:304: UserWarning: With less than 10 channels, multi-channel metrics might not be reliable.
  warnings.warn(


Compute : spike_locations + spike_amplitudes (workers: 8 processes fork):   0%|          | 0/1925 [00:00<?, ?i…

/home/halechr/FastData/Bapun/RatS/bapun_sess_init_scripts/.venv/lib/python3.11/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/halechr/FastData/Bapun/RatS/bapun_sess_init_scripts/.venv/lib/python3.11/site-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/home/halechr/FastData/Bapun/RatS/bapun_sess_init_scripts/.venv/lib/python3.11/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/home/halechr/FastData/Bapun/RatS/spikeinterface/src/spikeinterface/metrics/quality/misc_metrics.py:1026: UserWarning: Amplitude cutoff set to NaN for units [np.int64(9), np.int64(44), np.int64(111), np.int64(122), np.int64(158), np.int64(202), np.int64(245), np.int64(253)]: too few spikes (< 500).
  warnings.warn(
/home/halech

ValueError: Found array with 0 sample(s) (shape=(0, 37)) while a minimum of 1 is required by SimpleImputer.

## Good-unit labeling standards

UnitRefine SUA labels align with Phy manual `good` (17/138). The strict `probability > 0.8` gate was reducing SUA to 5.

- **phy**: match manual Phy curation
- **sua**: all model SUA (no probability gate)
- **sua_relaxed_prob** (default): SUA with `probability > PROB_THRESHOLD_DEFAULT` (0.65)
- **sua_high_conf**: SUA with `probability > 0.8` (original notebook behavior)
- **hybrid_phy_sua**: union of Phy good and model SUA
- **qm_and_sua**: SUA that also pass quality-metric thresholds


In [ ]:
phy_good_ids = sorting.unit_ids[sorting.get_property("quality") == "good"]
sua_mask = all_labels["prediction"] == "sua"
sua_df = all_labels.loc[sua_mask]

print("--- Filter stage counts ---")
print(f"Total units: {len(all_labels)}")
print(f"Phy good: {len(phy_good_ids)}")
print(f"Model SUA: {int(sua_mask.sum())}")
for t in PROB_CUTOFFS:
    sel = (all_labels["prediction"] == "sua") & (all_labels["probability"] > t)
    ids = set(all_labels.index[sel])
    overlap = len(set(phy_good_ids) & ids)
    print(f"  SUA & prob>{t}: {sel.sum()} (overlap with Phy good: {overlap})")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sua_df["probability"], bins=20, edgecolor="k", alpha=0.85)
for t in PROB_CUTOFFS:
    ax.axvline(t, linestyle="--", linewidth=1.2, label=f"prob>{t}")
ax.set_xlabel("SUA classifier probability")
ax.set_ylabel("count")
ax.set_title(f"SUA units (n={len(sua_df)}) vs Phy good (n={len(phy_good_ids)})")
ax.legend()
plt.tight_layout()
plt.show()

borderline = comparison.query("prediction == 'sua' & probability < @PROB_THRESHOLD_HIGH")
print(f"SUA with prob < {PROB_THRESHOLD_HIGH} (borderline): {len(borderline)}")
display(borderline[["prediction", "probability", "phy_label"]].sort_values("probability"))


In [ ]:
import spikeinterface.curation as sc

metrics_df = sorting_analyzer.get_extension("quality_metrics").get_data()
qm_thresholds = {"snr": {"greater": 5}, "firing_rate": {"greater": 0.1, "less": 200}, "isi_violations_ratio": {"less": 0.5}}
qm_labels = compute_qm_labels(sorting_analyzer, qm_thresholds)
print("Quality-metric threshold labels:")
print(qm_labels["qm_good"].value_counts())

try:
    bc_labels = sc.bombcell_label_units(sorting_analyzer=sorting_analyzer)
    print("\nBombcell labels:")
    print(bc_labels["Bombcell_label"].value_counts())
except Exception as exc:
    print(f"Bombcell skipped: {exc}")
    bc_labels = None


## Upstream quality (optional)

If you need more units than curation thresholds allow:
1. Mark additional clusters as `good` in Phy.
2. Merge over-split units (`compute_merge_unit_groups`), then rebuild the analyzer and re-run labeling.
3. Re-sort with CAR + motion correction, then Phy + this curation block.


In [ ]:
# OPTIONAL: preprocessing + re-sort + merge (uncomment blocks as needed)
# recording_preprocessed = si.bandpass_filter(si.common_reference(recording, reference="global", operator="median"))
# motion_folder = sorting_outputs_folder.joinpath("motion")
# recording_corrected = si.correct_motion(recording_preprocessed, preset="nonrigid_accurate", folder=motion_folder)
# sorting_rerun = run_sorter(sorter_name="spykingcircus2", recording=recording_corrected, folder=sorting_outputs_folder.joinpath("folder_SC2_preprocessed"), remove_existing_folder=True, delete_output_folder=True)

# merge_groups, extra = sc.compute_merge_unit_groups(sorting_analyzer, preset="similarity_correlograms")
# print(f"Suggested merge groups: {len(merge_groups)}")
# for g in merge_groups[:10]:
#     print(g)


## SpikeInterface GUI (optional)

Interactive inspection and manual curation with [SpikeInterface-GUI](https://spikeinterface-gui.readthedocs.io/en/latest/).

- Run the curation pipeline cell (`run-phy-model`) first so `sorting_analyzer` exists.
- Set `LAUNCH_SPIKEINTERFACE_GUI = True` in the cell below to open the desktop Qt viewer.
- The kernel blocks until you close the GUI window.
- CLI alternative: `uv run sigui <phy_sorting_analyzer_folder>` (see `result.paths.phy_sorting_analyzer_folder`).

In [ ]:
LAUNCH_SPIKEINTERFACE_GUI = False  # set True to open the desktop viewer

if LAUNCH_SPIKEINTERFACE_GUI:
    from spikeinterface_gui import run_mainwindow
    from spikeinterface_pipeline.analyzer import phy_job_kwargs

    job_kwargs = phy_job_kwargs(n_jobs=8)
    if not sorting_analyzer.has_extension("unit_locations"):
        sorting_analyzer.compute("unit_locations", method="monopolar_triangulation")
    if not sorting_analyzer.has_extension("template_similarity"):
        sorting_analyzer.compute("template_similarity", **job_kwargs)
    if not sorting_analyzer.has_extension("isi_histograms"):
        sorting_analyzer.compute("isi_histograms")

    run_mainwindow(sorting_analyzer, mode="desktop", curation=True, with_traces=True)